# 04.R5 — ResNet-50 Inference & 2048-d Feature Export

**Phase 04 — Document Processing & OCR (ResNet-50 Visual Pipeline)**  
**Objective:** Load the saved `models/phase_04/resnet50/best_model.pt` checkpoint, run batch inference across all inventoried documents, and export documented 2048-d visual features (`resnet50_f0000..resnet50_f2047`) plus predicted category and confidence to `data/processed/features/resnet50_document_embeddings.parquet` and `resnet50_inference_summary.csv`.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

MANIFEST_CSV = paths.interim_doc_proc / 'resnet50' / 'resnet50_dataset_manifest.csv'
MODEL_DIR = paths.resnet50_model_dir
BEST_CKPT = MODEL_DIR / 'best_model.pt'
LABEL_MAP_JSON = MODEL_DIR / 'label_mapping.json'

FEATURES_DIR = paths.processed_dir / 'features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET = FEATURES_DIR / 'resnet50_document_embeddings.parquet'
OUT_SUMMARY_CSV = FEATURES_DIR / 'resnet50_inference_summary.csv'

if not BEST_CKPT.exists():
    raise FileNotFoundError(f"Best checkpoint missing: {BEST_CKPT}. Run 03_resnet50_feature_extraction.ipynb first.")

df_all = pd.read_csv(MANIFEST_CSV)
label_info = json.loads(LABEL_MAP_JSON.read_text(encoding='utf-8'))
idx2label = {int(k): v for k, v in label_info['idx2label'].items()}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class InferDS(Dataset):
    def __init__(self, df, root, tf):
        self.df, self.root, self.tf = df.reset_index(drop=True), root, tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        im = ImageOps.exif_transpose(Image.open(self.root / r['relative_path'])).convert('RGB')
        return self.tf(im), r['document_id'], r['filename'], r['split']

ckpt = torch.load(str(BEST_CKPT), map_location=DEVICE)
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, label_info['num_classes'])
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE).eval()

embs_list = []
def _hook(_m, inp, _out):
    embs_list.append(inp[0].detach().cpu().numpy())
h = model.fc.register_forward_hook(_hook)

meta_rows = []
loader = DataLoader(InferDS(df_all, paths.root, eval_tf), batch_size=16, shuffle=False, num_workers=0)
with torch.no_grad():
    for imgs, doc_ids, fnames, splits in loader:
        probs = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
        pred_ids = probs.argmax(axis=1)
        confs = probs.max(axis=1)
        for did, fn, sp, pid, cf in zip(doc_ids, fnames, splits, pred_ids, confs):
            meta_rows.append({
                'document_id': did,
                'filename': fn,
                'split': sp,
                'predicted_category': idx2label[int(pid)],
                'prediction_confidence': round(float(cf), 4),
            })
h.remove()

emb_matrix = np.vstack(embs_list)
feat_cols = [f'resnet50_f{i:04d}' for i in range(emb_matrix.shape[1])]
df_meta = pd.DataFrame(meta_rows)
df_feats = pd.concat([df_meta, pd.DataFrame(emb_matrix, columns=feat_cols)], axis=1)

df_meta.to_csv(OUT_SUMMARY_CSV, index=False, encoding='utf-8')
df_feats.to_parquet(OUT_PARQUET, index=False)

print(f"[OK] Exported {len(df_feats):,} x {emb_matrix.shape[1]}-d ResNet-50 embeddings -> {OUT_PARQUET}")
print(f"[OK] Exported inference summary -> {OUT_SUMMARY_CSV}")
display(df_meta.head(10))
